In [1]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os

OUTPUT_DIR = "/content/drive/MyDrive/Customer_Churn_ML_Outputs"

os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Output folder created:")
print(OUTPUT_DIR)

Output folder created:
/content/drive/MyDrive/Customer_Churn_ML_Outputs


In [8]:
# Create your output folders
MODEL_DIR = f"{OUTPUT_DIR}/models"
PREDICTION_DIR = f"{OUTPUT_DIR}/predictions"
METRIC_DIR = f"{OUTPUT_DIR}/metrics"

os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(PREDICTION_DIR, exist_ok=True)
os.makedirs(METRIC_DIR, exist_ok=True)

print("Output folders ready.")

Output folders ready.


In [9]:
# Import libraries
import os
import joblib
import numpy as np
import pandas as pd


from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay,
)

print("Libraries imported successfully.")

Libraries imported successfully.


In [26]:
# Create a reusable evaluation function

def evaluate_model(model, model_name, X_test, y_test):

    # Predictions
    y_pred = model.predict(X_test)

    # Probability of Churn = 1
    y_prob = model.predict_proba(X_test)[:, 1]

    # Metrics
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, zero_division=0)
    recall = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    roc_auc = roc_auc_score(y_test, y_prob)

    # Confusion matrix
    cm = confusion_matrix(y_test, y_pred)

    # Classification report
    report = classification_report(
        y_test,
        y_pred,
        zero_division=0
    )

    # Print results
    print("*" * 60)
    print(model_name)
    print("*" * 60)


    print(f"Accuracy      : {accuracy:.4f}")
    print(f"Precision     : {precision:.4f}")
    print(f"Recall        : {recall:.4f}")
    print(f"F1 Score      : {f1:.4f}")
    print(f"ROC-AUC       : {roc_auc:.4f}")

    print("\nClassification Report:")
    print(report)

    print("Confusion Matrix:")
    print(cm)

    # Save predictions
    predictions_df = pd.DataFrame({
        "actual_churn": y_test.values,
        "predicted_churn": y_pred,
        "predicted_probability": y_prob
    })

    prediction_path = os.path.join(
        PREDICTION_DIR,
        f"{model_name.lower().replace(' ', '_')}_predictions.csv"
    )

    predictions_df.to_csv(prediction_path, index=False)

    # Save metrics
    metrics_df = pd.DataFrame({
        "Model": [model_name],
        "Accuracy": [accuracy],
        "Precision": [precision],
        "Recall": [recall],
        "F1_Score": [f1],
        "ROC_AUC": [roc_auc]
    })

    metrics_path = os.path.join(
        METRIC_DIR,
        f"{model_name.lower().replace(' ', '_')}_metrics.csv"
    )

    metrics_df.to_csv(metrics_path, index=False)


    print(f"\nPredictions saved to:")
    print(prediction_path)

    print("\nMetrics saved to:")
    print(metrics_path)


    return {
        "Model": model_name,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1_Score": f1,
        "ROC_AUC": roc_auc
    }

In [27]:
# Create an empty list for all model results
model_results = []

In [28]:
import pandas as pd

# Training data
X_train = pd.read_csv("training_data_preprocessed.csv")
y_train = pd.read_csv("training_data.csv")["Churn"]

# Testing data
X_test = pd.read_csv("testing_data_preprocessed.csv")
y_test = pd.read_csv("testing_data.csv")["Churn"]

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_test :", X_test.shape)
print("y_test :", y_test.shape)

X_train: (352665, 15)
y_train: (352665,)
X_test : (88167, 15)
y_test : (88167,)


In [29]:
# Train Model 1: Logistic Regression
logistic_model = LogisticRegression(
    max_iter=1000,
    random_state=42,
    n_jobs=-1
)

logistic_model.fit(X_train, y_train)

LogisticRegression(max_iter=1000, n_jobs=-1, random_state=42)

In [30]:
logistic_result = evaluate_model(
    logistic_model,
    "Logistic Regression",
    X_test,
    y_test
)

model_results.append(logistic_result)

************************************************************
Logistic Regression
************************************************************
Accuracy      : 0.8934
Precision     : 0.9234
Recall        : 0.8855
F1 Score      : 0.9040
ROC-AUC       : 0.9590

Classification Report:
              precision    recall  f1-score   support

         0.0       0.86      0.90      0.88     38167
         1.0       0.92      0.89      0.90     50000

    accuracy                           0.89     88167
   macro avg       0.89      0.89      0.89     88167
weighted avg       0.89      0.89      0.89     88167

Confusion Matrix:
[[34492  3675]
 [ 5726 44274]]

Predictions saved to:
/content/drive/MyDrive/Customer_Churn_ML_Outputs/predictions/logistic_regression_predictions.csv

Metrics saved to:
/content/drive/MyDrive/Customer_Churn_ML_Outputs/metrics/logistic_regression_metrics.csv


In [31]:
logistic_model_path = os.path.join(
    MODEL_DIR,
    "logistic_regression.joblib"
)

joblib.dump(logistic_model, logistic_model_path)

print("Logistic Regression model saved:")
print(logistic_model_path)

Logistic Regression model saved:
/content/drive/MyDrive/Customer_Churn_ML_Outputs/models/logistic_regression.joblib


In [32]:
# Train Model 2: Decision Tree
decision_tree_model = DecisionTreeClassifier(
    random_state=42
)

decision_tree_model.fit(X_train, y_train)

DecisionTreeClassifier(random_state=42)

In [33]:
decision_tree_result = evaluate_model(
    decision_tree_model,
    "Decision Tree",
    X_test,
    y_test
)

model_results.append(decision_tree_result)

************************************************************
Decision Tree
************************************************************
Accuracy      : 0.9999
Precision     : 0.9999
Recall        : 0.9999
F1 Score      : 0.9999
ROC-AUC       : 0.9999

Classification Report:
              precision    recall  f1-score   support

         0.0       1.00      1.00      1.00     38167
         1.0       1.00      1.00      1.00     50000

    accuracy                           1.00     88167
   macro avg       1.00      1.00      1.00     88167
weighted avg       1.00      1.00      1.00     88167

Confusion Matrix:
[[38164     3]
 [    5 49995]]

Predictions saved to:
/content/drive/MyDrive/Customer_Churn_ML_Outputs/predictions/decision_tree_predictions.csv

Metrics saved to:
/content/drive/MyDrive/Customer_Churn_ML_Outputs/metrics/decision_tree_metrics.csv


In [34]:
decision_tree_model_path = os.path.join(
    MODEL_DIR,
    "decision_tree.joblib"
)

joblib.dump(decision_tree_model, decision_tree_model_path)

print("Decision Tree model saved:")
print(decision_tree_model_path)

Decision Tree model saved:
/content/drive/MyDrive/Customer_Churn_ML_Outputs/models/decision_tree.joblib


In [35]:
# Train Model 3: Random Forest
random_forest_model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

random_forest_model.fit(X_train, y_train)

RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=42)

In [36]:
random_forest_result = evaluate_model(
    random_forest_model,
    "Random Forest",
    X_test,
    y_test
)

model_results.append(random_forest_result)

************************************************************
Random Forest
************************************************************
Accuracy      : 0.9993
Precision     : 0.9999
Recall        : 0.9988
F1 Score      : 0.9994
ROC-AUC       : 1.0000

Classification Report:
              precision    recall  f1-score   support

         0.0       1.00      1.00      1.00     38167
         1.0       1.00      1.00      1.00     50000

    accuracy                           1.00     88167
   macro avg       1.00      1.00      1.00     88167
weighted avg       1.00      1.00      1.00     88167

Confusion Matrix:
[[38164     3]
 [   59 49941]]

Predictions saved to:
/content/drive/MyDrive/Customer_Churn_ML_Outputs/predictions/random_forest_predictions.csv

Metrics saved to:
/content/drive/MyDrive/Customer_Churn_ML_Outputs/metrics/random_forest_metrics.csv


In [37]:
random_forest_model_path = os.path.join(
    MODEL_DIR,
    "random_forest.joblib"
)

joblib.dump(random_forest_model, random_forest_model_path)

print("Random Forest model saved:")
print(random_forest_model_path)

Random Forest model saved:
/content/drive/MyDrive/Customer_Churn_ML_Outputs/models/random_forest.joblib


In [38]:
# Train Model 4: Gradient Boosting
gradient_boosting_model = GradientBoostingClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    random_state=42
)

gradient_boosting_model.fit(X_train, y_train)

GradientBoostingClassifier(random_state=42)

In [39]:
gradient_boosting_result = evaluate_model(
    gradient_boosting_model,
    "Gradient Boosting",
    X_test,
    y_test
)

model_results.append(gradient_boosting_result)

************************************************************
Gradient Boosting
************************************************************
Accuracy      : 0.9967
Precision     : 0.9999
Recall        : 0.9943
F1 Score      : 0.9971
ROC-AUC       : 0.9999

Classification Report:
              precision    recall  f1-score   support

         0.0       0.99      1.00      1.00     38167
         1.0       1.00      0.99      1.00     50000

    accuracy                           1.00     88167
   macro avg       1.00      1.00      1.00     88167
weighted avg       1.00      1.00      1.00     88167

Confusion Matrix:
[[38164     3]
 [  287 49713]]

Predictions saved to:
/content/drive/MyDrive/Customer_Churn_ML_Outputs/predictions/gradient_boosting_predictions.csv

Metrics saved to:
/content/drive/MyDrive/Customer_Churn_ML_Outputs/metrics/gradient_boosting_metrics.csv


In [40]:
gradient_boosting_model_path = os.path.join(
    MODEL_DIR,
    "gradient_boosting.joblib"
)

joblib.dump(gradient_boosting_model, gradient_boosting_model_path)

print("Gradient Boosting model saved:")
print(gradient_boosting_model_path)

Gradient Boosting model saved:
/content/drive/MyDrive/Customer_Churn_ML_Outputs/models/gradient_boosting.joblib


In [41]:
model_comparison = pd.DataFrame(model_results)

model_comparison

,Model,Accuracy,Precision,Recall,F1_Score,ROC_AUC
0,Logistic Regression,0.893373,0.923356,0.88548,0.904021,0.959026
1,Decision Tree,0.999909,0.999940,0.99990,0.999920,0.999911
2,Random Forest,0.999297,0.999940,0.99882,0.999380,0.999999
3,Gradient Boosting,0.996711,0.999940,0.99426,0.997092,0.999882


In [42]:
comparison_path = os.path.join(
    METRIC_DIR,
    "model_comparison.csv"
)

model_comparison.to_csv(
    comparison_path,
    index=False
)

print("Model comparison saved:")
print(comparison_path)

Model comparison saved:
/content/drive/MyDrive/Customer_Churn_ML_Outputs/metrics/model_comparison.csv


In [43]:
model_comparison_sorted = model_comparison.sort_values(
    by="F1_Score",
    ascending=False
)

model_comparison_sorted

,Model,Accuracy,Precision,Recall,F1_Score,ROC_AUC
1,Decision Tree,0.999909,0.999940,0.99990,0.999920,0.999911
2,Random Forest,0.999297,0.999940,0.99882,0.999380,0.999999
3,Gradient Boosting,0.996711,0.999940,0.99426,0.997092,0.999882
0,Logistic Regression,0.893373,0.923356,0.88548,0.904021,0.959026
